# Pipeline Analysis

Companion notebook for `pipeline.py`. Reads logs, the SQLite knowledge store, and attempt history — never calls pipeline functions directly.

**Set `PROJECT` below before running.**

In [ ]:
import json
import re
import sqlite3
import subprocess
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# ── Config ──────────────────────────────────────────────────────────────────
ROOT        = Path("/home/user/determ")
PROJECT     = "anyhow-master"          # change this to target project
LOG_FILE    = Path("/tmp/pipeline_anyhow.log")  # nohup log

project_dir = ROOT / PROJECT
run_log     = project_dir / "run.log"
attempts_log = project_dir / ".task0c_attempts.jsonl"
db_path     = project_dir / "pipeline_knowledge.sqlite"

print(f"Project : {project_dir}")
print(f"run.log : {'exists' if run_log.exists() else 'NOT FOUND'}")
print(f"attempts: {'exists' if attempts_log.exists() else 'NOT FOUND'}")
print(f"sqlite  : {'exists' if db_path.exists() else 'NOT FOUND'}")

---
## 1 · Pipeline status

In [ ]:
# Is the pipeline running?
result = subprocess.run(["pgrep", "-la", "python"], capture_output=True, text=True)
pipeline_procs = [l for l in result.stdout.splitlines() if "pipeline" in l]
if pipeline_procs:
    print("RUNNING:")
    for p in pipeline_procs:
        print(" ", p)
else:
    print("NOT running.")

# Coverage sentinel
sentinel = project_dir / ".coverage_augmented"
print(f"\n.coverage_augmented : {'EXISTS (TASK 0c done)' if sentinel.exists() else 'absent'}")

In [ ]:
# Tail the live nohup log (last 60 lines)
if LOG_FILE.exists():
    lines = LOG_FILE.read_text().splitlines()
    print("\n".join(lines[-60:]))
else:
    print(f"{LOG_FILE} not found.")

---
## 2 · TASK 0c — attempt history

In [ ]:
def load_attempts(path: Path) -> pd.DataFrame:
    rows = []
    if not path.exists():
        return pd.DataFrame()
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    return df

df = load_attempts(attempts_log)
if df.empty:
    print("No attempt data yet.")
else:
    print(f"{len(df)} total attempts across {df['file'].nunique()} file(s)")
    print("\nOutcome breakdown:")
    print(df["outcome"].value_counts().to_string())

In [ ]:
# Per-file attempt count and success rate
if not df.empty:
    summary = (
        df.groupby("file")
        .agg(
            attempts=("attempt", "count"),
            successes=("outcome", lambda x: (x == "success").sum()),
            last_outcome=("outcome", "last"),
        )
        .sort_values("attempts", ascending=False)
    )
    summary["success_rate"] = (summary["successes"] / summary["attempts"] * 100).round(1)
    display(summary)

In [ ]:
# Most common compile error patterns across all failures
if not df.empty:
    errors = df[df["outcome"] == "compile_error"]["compile_error"].dropna()
    if errors.empty:
        print("No compile errors recorded.")
    else:
        # Extract first meaningful error line from each entry
        error_lines = []
        for e in errors:
            for line in str(e).splitlines():
                line = line.strip()
                if line.startswith("error") or line.startswith("^"):
                    # Normalise: strip file paths and line numbers
                    line = re.sub(r'-->[^:]+:\d+:\d+', '', line)
                    line = re.sub(r'\s+', ' ', line).strip()
                    error_lines.append(line[:120])
                    break
        counts = Counter(error_lines).most_common(15)
        print("Top compile errors:")
        for msg, n in counts:
            print(f"  {n:3d}×  {msg}")

In [ ]:
# Outcome timeline chart
if not df.empty and "ts" in df.columns and df["ts"].notna().any():
    color_map = {
        "success": "#2ecc71",
        "compile_error": "#e74c3c",
        "no_block": "#f39c12",
        "reverted": "#9b59b6",
    }
    files = df["file"].unique()
    file_idx = {f: i for i, f in enumerate(sorted(files))}

    fig, ax = plt.subplots(figsize=(14, max(3, len(files) * 0.5)))
    for _, row in df.iterrows():
        color = color_map.get(row["outcome"], "#95a5a6")
        y = file_idx.get(row["file"], 0)
        ax.scatter(row["ts"], y, color=color, s=60, zorder=3)

    ax.set_yticks(range(len(files)))
    ax.set_yticklabels(sorted(files), fontsize=8)
    ax.set_xlabel("Time")
    ax.set_title("TASK 0c attempt outcomes over time")
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=c, label=l) for l, c in color_map.items()],
              loc="upper left", fontsize=8)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## 3 · SQLite knowledge store

In [ ]:
def query_db(sql: str, params=()) -> pd.DataFrame:
    if not db_path.exists():
        print(f"DB not found: {db_path}")
        return pd.DataFrame()
    with sqlite3.connect(str(db_path)) as conn:
        try:
            return pd.read_sql_query(sql, conn, params=params)
        except Exception as e:
            print(f"Query error: {e}")
            return pd.DataFrame()

# Table sizes
for table in ["type_signals", "compile_patterns", "genesis_rules", "module_status"]:
    n = query_db(f"SELECT COUNT(*) as n FROM {table}")
    count = n.iloc[0]["n"] if not n.empty else "(table missing)"
    print(f"  {table:<20} {count} rows")

In [ ]:
# Type signals — which types were hardest to construct in tests
ts = query_db("""
    SELECT type_name, SUM(difficulty) as total_difficulty,
           COUNT(*) as occurrences, file
    FROM type_signals
    GROUP BY type_name
    ORDER BY total_difficulty DESC
    LIMIT 20
""")
if ts.empty:
    print("No type signals yet.")
else:
    display(ts)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(ts["type_name"][:15][::-1], ts["total_difficulty"][:15][::-1],
            color="#e74c3c")
    ax.set_xlabel("Difficulty score")
    ax.set_title("Types hardest to construct in tests (TASK 0c → TASK 1 signal)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Compile patterns — recurring error categories per module
cp = query_db("""
    SELECT module, error_category, COUNT(*) as count,
           MAX(ts) as last_seen
    FROM compile_patterns
    GROUP BY module, error_category
    ORDER BY count DESC
    LIMIT 30
""")
if cp.empty:
    print("No compile patterns yet.")
else:
    display(cp)

    pivot = cp.pivot_table(index="module", columns="error_category",
                           values="count", fill_value=0)
    if not pivot.empty:
        fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns)), max(3, len(pivot) * 0.6)))
        pivot.plot(kind="barh", stacked=True, ax=ax, colormap="tab10")
        ax.set_title("Compile error categories by module (TASK 2 fix loop)")
        ax.set_xlabel("Occurrences")
        plt.tight_layout()
        plt.show()

In [ ]:
# Genesis rules — structured rules extracted by TASK 4
gr = query_db("""
    SELECT category, module_scope, rule_text, iteration, ts
    FROM genesis_rules
    ORDER BY id DESC
    LIMIT 40
""")
if gr.empty:
    print("No genesis rules yet.")
else:
    pd.set_option("display.max_colwidth", 120)
    display(gr)

In [ ]:
# Module status — pass count trend over iterations
ms = query_db("""
    SELECT module, iteration, outcome, pass_count, fail_count, ts
    FROM module_status
    ORDER BY module, iteration
""")
if ms.empty:
    print("No module status yet.")
else:
    display(ms)

    fig, ax = plt.subplots(figsize=(12, 4))
    for module, grp in ms.groupby("module"):
        ax.plot(grp["iteration"], grp["pass_count"], marker="o", label=module)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Passing tests")
    ax.set_title("Test pass count per module over iterations")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## 4 · run.log analysis

In [ ]:
def parse_run_log(path: Path) -> pd.DataFrame:
    """Extract timestamped key events from run.log."""
    if not path.exists():
        return pd.DataFrame()
    rows = []
    ts_re = re.compile(r'\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s*(.*)')
    for line in path.read_text().splitlines():
        m = ts_re.match(line)
        if not m:
            continue
        ts, msg = m.group(1), m.group(2)
        # Classify event
        if "Coverage:" in msg:
            pct_m = re.search(r'Coverage:\s*([\d.]+)%', msg)
            file_m = re.search(r'\[([^\]]+)\]', msg)
            rows.append({"ts": ts, "event": "coverage",
                         "module": file_m.group(1) if file_m else "",
                         "value": float(pct_m.group(1)) if pct_m else None,
                         "msg": msg})
        elif "context-guard" in msg:
            rows.append({"ts": ts, "event": "context_guard",
                         "module": "", "value": None, "msg": msg})
        elif "TASK" in msg and any(f"TASK {n}" in msg for n in ["0b","0c","1","2","3","4"]):
            rows.append({"ts": ts, "event": "task_start",
                         "module": "", "value": None, "msg": msg})
        elif "100% coverage" in msg:
            rows.append({"ts": ts, "event": "100pct",
                         "module": "", "value": 100.0, "msg": msg})
        elif "compile error" in msg.lower() or "Compile error" in msg:
            rows.append({"ts": ts, "event": "compile_error",
                         "module": "", "value": None, "msg": msg})
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df["ts"] = pd.to_datetime(df["ts"])
    return df

log_df = parse_run_log(run_log)
if log_df.empty:
    print("run.log not found or no parseable events yet.")
else:
    print(f"{len(log_df)} events parsed.")
    print(log_df["event"].value_counts().to_string())

In [ ]:
# Coverage % over time (from run.log)
if not log_df.empty:
    cov = log_df[log_df["event"] == "coverage"].dropna(subset=["value"])
    if cov.empty:
        print("No coverage events in log yet.")
    else:
        fig, ax = plt.subplots(figsize=(12, 4))
        for module, grp in cov.groupby("module"):
            ax.plot(grp["ts"], grp["value"], marker="o", label=module)
        ax.axhline(100, color="green", linestyle="--", alpha=0.5, label="100%")
        ax.set_ylabel("Coverage %")
        ax.set_title("Coverage progress over time (TASK 0c)")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

In [ ]:
# Context-guard events — shows when prompts exceeded the budget
if not log_df.empty:
    guard_events = log_df[log_df["event"] == "context_guard"]
    if guard_events.empty:
        print("No context-guard events (good — budget was never exceeded).")
    else:
        print(f"{len(guard_events)} context-guard events:")
        for _, row in guard_events.iterrows():
            print(f"  {row['ts']}  {row['msg'][:120]}")

---
## 5 · Prompt prototyping

Use these cells to test prompt variations against Ollama before wiring them into `pipeline.py`.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MODEL = os.getenv("MODEL", "qwen3:32b")
OLLAMA_NUM_CTX = int(os.getenv("OLLAMA_NUM_CTX", "32768"))

print(f"Model  : {MODEL}")
print(f"Host   : {OLLAMA_HOST}")
print(f"Ctx    : {OLLAMA_NUM_CTX}")

In [ ]:
import requests

def proto_ask(prompt: str, model: str = MODEL, num_ctx: int = OLLAMA_NUM_CTX,
              think: bool = False) -> str:
    """
    Send a prompt to Ollama and return the response text.
    Set think=False to suppress <think>...</think> blocks in the output.
    """
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"num_ctx": num_ctx},
    }
    resp = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=300)
    resp.raise_for_status()
    text = resp.json()["response"]
    if not think:
        text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    return text

print("proto_ask() ready.")

In [ ]:
# Example: test a prompt against a source file
# Edit the prompt below and run the cell

target_file = project_dir / "src" / "error.rs"   # adjust path

if target_file.exists():
    source = target_file.read_text()
    prompt = f"""List the public types defined in this Rust file and how to construct each one:

```rust
{source[:3000]}
```
"""
    response = proto_ask(prompt)
    print(response)
else:
    print(f"{target_file} not found — update the path above.")

---
## 6 · Quick inspection helpers

In [ ]:
# Print a specific file's attempt history in full
TARGET_FILE = "src/error.rs"   # change this

if not df.empty:
    rows = df[df["file"] == TARGET_FILE].to_dict("records")
    if not rows:
        print(f"No attempts found for {TARGET_FILE}")
    for r in rows:
        print(f"Attempt {r['attempt']} — {r['outcome']} — {r['ts']}")
        if r.get("compile_error"):
            print("  Compile error:")
            print("  " + "\n  ".join(str(r['compile_error'])[:800].splitlines()))
        print()

In [ ]:
# Show current coverage_strategy.md
strat = project_dir / "coverage_strategy.md"
if strat.exists():
    print(strat.read_text())
else:
    print("coverage_strategy.md not generated yet.")

In [ ]:
# Show current genesis.md (if it exists)
genesis = project_dir / "genesis.md"
if genesis.exists():
    print(genesis.read_text())
else:
    print("genesis.md not generated yet.")

In [ ]:
# Diff two genesis.md snapshots saved in tests_output_N.txt files
import difflib

OUTPUT_SEP = "=== test output ==="

def read_genesis_from_output(n: int) -> str:
    p = project_dir / f"tests_output_{n}.txt"
    if not p.exists():
        return ""
    content = p.read_text()
    idx = content.find(OUTPUT_SEP)
    return content[:idx].strip() if idx != -1 else content.strip()

N_BEFORE = 1   # change these iteration numbers
N_AFTER  = 2

before = read_genesis_from_output(N_BEFORE).splitlines(keepends=True)
after  = read_genesis_from_output(N_AFTER).splitlines(keepends=True)

if not before and not after:
    print("No tests_output_N.txt files found yet.")
else:
    diff = list(difflib.unified_diff(before, after,
                                     fromfile=f"genesis iter {N_BEFORE}",
                                     tofile=f"genesis iter {N_AFTER}"))
    if diff:
        print("".join(diff[:100]))
    else:
        print("No changes between these iterations.")